## 답변에 필요한 도구를 순서대로 활용했는가? (Trajectory Evaluation)

### Trajectory 평가란?
단순히 "올바른 도구를 사용했는가?"를 넘어, "올바른 순서로 사용했는가?" 를 검증

예시: "논문 검색 후 이메일 전송" 작업

✅ 올바른 궤적: arxiv → send_gmail_message
❌ 잘못된 궤적: send_gmail_message → arxiv (검색 전에 이메일 전송)
❌ 잘못된 궤적: arxiv만 실행 (이메일 전송 누락)

* 평가 프로세스
```
테스트 데이터셋 (LangSmith)
  • query: "논문 검색 후 이메일 전송"
  • metadata.target_tool_sequence: ["tavily_search", "send_gmail_message"]
        │
        ▼
┌─────────────────────────────────┐
│   run_agent_to_completion       │  ← 에이전트 실행
└─────────────────────────────────┘
        │
        ▼
┌─────────────────────────────────┐
│      check_tool_sequence        │
│  • 실제 도구 호출 순서 추출           │
│  • 기대 순서와 비교                 │
└─────────────────────────────────┘
        │
        ▼
    평가 결과 (1 또는 0)
```


* 평가 내용

|평가 방식|검증 내용|예시|
|-------|------|----|
|답변 정확성|최종 답변이 정답과 일치하는가|LLM-as-Judge|
|도구 선택|올바른 도구를 선택했는가|check_target_tool|
|궤적 평가|올바른 순서로 도구를 실행했는가|check_tool_sequence|


In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langsmith import Client

# LangSmith 클라이언트 - 데이터셋 관리, 실험 실행, 결과 조회에 사용
ls_client = Client()

In [3]:
dataset = ls_client.read_dataset(dataset_name="trajectory-evaluation-dataset")

### 1. 에이전트 실행 함수 정의

In [4]:
from multi_tool_agent import agent

def run_agent_to_completion(inputs: dict) -> dict:
    """
    에이전트를 실행하여 최종 결과를 반환하는 함수
    
    Args:
        inputs (dict): 에이전트에 입력할 데이터 (예: {"question": "What is the capital of France?"})
    
    Returns:
        dict: 에이전트 실행 결과 (messages 포함)
    """
    result = agent.invoke({
        "messages": [{"role": "user", "content": inputs['question']}]
    })
    print(f'result in run agent: {result}')
    return result

### 2. 유틸리티 함수: 연속 중복 제거
* 에이전트가 도구 호출에 실패하면 같은 도구를 재시도할 수 있음
    * 예: ['search', 'search', 'email'] → ['search', 'email']

* 이런 재시도는 궤적 평가에서 오탐(false negative)을 유발할 수 있으므로, 연속된 중복을 제거하는 기능 정의

In [5]:
def remove_consecutive_duplicates(sequence: list) -> list:
    """연속된 중복 도구 호출(재시도) 제거.
    
    에이전트가 실패 후 같은 도구를 재시도하는 경우를 처리합니다.
    예: ['search', 'search', 'email'] → ['search', 'email']
    """
    if not sequence:
        return []
    result = [sequence[0]]
    for item in sequence[1:]:
        if item != result[-1]:
            result.append(item)
    return result

### 3.Evaluator 정의: 도구 호출 순서 확인
* 이 evaluator는 "올바른 순서로 도구를 사용했는가?"를 평가함

* 평가 로직:
    1. 에이전트의 메시지에서 ToolMessage를 추출하여 도구 호출 순서 생성
    2. 연속 중복 제거 (재시도 처리)
    3. 기대 순서(metadata.target_tool_sequence)와 길이 비교
    4. 각 위치의 도구명이 매칭되는지 확인 (부분 문자열 매칭 허용)


In [6]:
from langchain_core.messages import ToolMessage

def check_tool_sequence(inputs, outputs, reference_outputs, example):
    """에이전트가 올바른 순서로 도구를 실행했는지 확인하는 evaluator.
    
    Args:
        example: LangSmith Example - metadata['target_tool_sequence']에 기대 순서가 기록됨
                 예: ["tavily_search", "send_gmail_message"]
    
    Returns:
        int: 1(순서 일치) 또는 0(순서 불일치 또는 도구 누락)
    """
    # Golden Dataset에 기록된 기대 도구 호출 순서
    expected_sequence = example.metadata['target_tool_sequence']

    messages = outputs.get('messages', [])
    # 에이전트의 실제 도구 호출 순서 추출
    raw_sequence = [msg.name for msg in messages if isinstance(msg, ToolMessage)]
    # 재시도로 인한 연속 중복 제거
    unique_sequence = remove_consecutive_duplicates(raw_sequence)

    # 도구 호출 횟수가 다르면 즉시 실패
    if len(unique_sequence) != len(expected_sequence):
        return 0

    # 각 위치의 도구명 비교 (부분 문자열 매칭 허용)
    # 예: expected="arxiv", actual="arxiv_search" → 매칭 성공
    return int(all(
        exp in act or act in exp
        for exp, act in zip(expected_sequence, unique_sequence)
    ))

In [7]:
experiment_results = ls_client.evaluate(
    run_agent_to_completion,              # 평가 대상 함수
    data="trajectory-evaluation-dataset", # 궤적 평가 전용 데이터셋
    evaluators=[check_tool_sequence],     # 도구 순서 evaluator
    max_concurrency=1,                    # 순차 실행 (디버깅에 유리)
    num_repetitions=1,                    # 각 예제 1회 실행
)

/Users/a202304035/LLM_Eval_study/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'left-point-23' at:
https://smith.langchain.com/o/2f003b67-27f4-48d0-8856-e1db2397e3d4/datasets/792a5a6f-1466-45bf-989d-1d0eea887d3c/compare?selectedSessions=65aa2d50-92d9-4ce0-88c5-7a2105748155




0it [00:00, ?it/s]Python REPL can execute arbitrary code. Use with caution.
1it [00:08,  8.74s/it]

result in run agent: {'messages': [HumanMessage(content='피보나치 수열의 20번째 값을 Python으로 계산해줘', additional_kwargs={}, response_metadata={}, id='209ef29e-88e9-4f6c-8c20-fdae4a41c7cc'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'python_repl_tool', 'arguments': '{"command": "def fibonacci(n):\\n    if n <= 0: return 0\\n    elif n == 1: return 1\\n    a, b = 0, 1\\n    for _ in range(2, n + 1):\\n        a, b = b, a + b\\n    return b\\n\\nprint(\\"20\\ubc88\\uc9f8 \\ud53c\\ubcf4\\ub098\\uce58 \\uc218:\\", fibonacci(20))"}'}, '__gemini_function_call_thought_signatures__': {'0af5802a-02c0-4094-b16d-3bb31cbcccdf': 'ErAHCq0HAQw51sdzGvpOQE6nIAOYOi5BotV8Or6pWxCJ6fv5TKG7/S2bmQNXBtaxazJmr64h7q7WOnfsXTx3NEgGWlsRo4qOG7QfOqtsDBno6Q2H1qIHsGuKuBqpdm+gDbqwGFowGIl9NPmCzDW/QSnMvpAhjchKi/PIcMUumezamymgxVDPZg3vv13+ck/aDirofhPnC03LX6XDHcL68IPto9dr0VOF3+xvxCkaUpY5+t6i/7YGR+A55Z4UGweCRrhrVpuLMCoI4EjsEyq+vGgiSkggD79YOs6wZ5Zo+a9QCdndNEiP7Hh/TgBdwUyn66lsiz0YDc6bUsnhPjBsepmiVwptdBHjh3/ICuXw8aN

2it [00:49, 27.29s/it]

...

6it [02:44, 32.26s/it]

result in run agent: {'messages': [HumanMessage(content='2024년 AI 트렌드를 웹에서 검색하고 Python으로 주요 키워드 빈도를 분석해줘', additional_kwargs={}, response_metadata={}, id='a0bd0544-1a7a-4a73-b376-d524ae801f5b'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'tavily_search', 'arguments': '{"query": "2024\\ub144 AI \\ud2b8\\ub80c\\ub4dc"}'}, '__gemini_function_call_thought_signatures__': {'2a5e9499-5c71-4833-841f-08f4e88282a7': 'Er4HCrsHAQw51sdfFr0kAZIlajJ8BMmowE0U7cRBk0X2xMH3EZzKY0Fpu94Qtw6vYF2mxKYZbI+tKhXmsNKpxTzCxHTGv/xWYfH85MmZYLsB2qefuBqd+4+Pp9JOBL+uagKg1wzRLUXYVkSxOaDW9K3RkzLiDTcLf0MoJ9wz/0AJlkHghNVqaqV2mMimJum00rF2ErYh/6dPAVjE5xtI8jXMZQEf3NmfZ/T0HHb0oY+ICMAE1k113Mah+QsOBHy8Pg03lTW80nTy9YuahlsyzdnPIdJkK043IOWTx7IC1K2kYg/UJoETCt4pAapXTND5m2sDSUUjC713SsTbAA3rxBx7EYQJ4g1Q0q7bY3o6It/Suakzv4KrhfjxQ8v9LXz3+xt+/emt7ZxfSTv7ORBO+vYkLvpQKpqHYQnQiSyaGrf4POvEzQzFAUqO7F+GBRm1ZAZFN+3aIh331LK3nurD64BTOHhw7IDOAFMIT4ptx7F7QfM1cmPgGkbOZ6gU/HzJtwEgv5DiJZ6WfJRA8IgxVnMMSGpOL0dsJsYlAIyMYlYyQmKYnSkT2

7it [03:11, 30.38s/it]

result in run agent: {'messages': [HumanMessage(content='Transformer 아키텍처 관련 논문을 arXiv에서 찾아서 요약을 드래프트로 저장해줘. 받는 사람은 iris122413@gmail.com', additional_kwargs={}, response_metadata={}, id='e614bc42-dfa5-4a98-ac08-96ed69774d86'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'arxiv', 'arguments': '{"query": "Transformer architecture"}'}, '__gemini_function_call_thought_signatures__': {'e6391a6b-7dae-4c9c-b1d1-14d945becb0d': 'EoIJCv8IAQw51sfAV54R1ZgaGnjKpn9DkFCKhRgkG6XtPaDFaTR99xnF4IFDWwht3fxLiIF4Qd8CtZcDcATvPYfX2Ww8VD6Ga7ALqFs6ChTkplvgPIroeEAdBE4zr5Dmo0EnmxVersTMr6ah5zvkQV76t8uzdGFbkHY5c1Mku4kdlq3et7MAloWMYcZfSywe2mSECuhwKMYYLwyx705sQZmB7gyWQJHI4rqS27jy7fV1Y5lFVtb0lAosMALzVSLWCfMRczMC6K/bg8VbUiCBOq9yBJD6HdSTMsTFNLIzNjXe1X/DeQLOX2Mw+1s51S8wYVfzyOoMHg83y90KkSlibSPyibKL3HoASd/QZfXc4VgfSNuE1Ky+FD+4yLAQmDguFY60icHl8iWT1DKLpbNo36d9QqnLeIhUq6cLkplCxp/q+gs9i1B5GWmBWrejtgjMFGTZiSQ8Q7qty7lXBW+i0LGpsEjmrArrkVvlNigdDlX4D+EKU1gzZX9ZB+nKXOg2HYVZrIvgUpaMtygc4ZUj4WH2RVacF0jCWj+Y8Dk+f

8it [03:11, 23.99s/it]
